In [0]:
storage_account = "insurancestorage01"

spark.conf.set(
"fs.azure.account.key."+storage_account+".dfs.core.windows.net",
"azure_Storage_key"
)

spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

bronze_adls_path = "abfss://bronze@"+storage_account+".dfs.core.windows.net/customer_enrollment/"
silver_adls_path = "abfss://silver@"+storage_account+".dfs.core.windows.net/customer_enrollment/"

In [0]:
from pyspark.sql.types import *

enrollment_schema = StructType([
    StructField("customer_policy_id", LongType(), True),
    StructField("policy_number", StringType(), True),
    StructField("customer_id", LongType(), True),
    StructField("policy_product_id", LongType(), True),
    StructField("enrollment_date", DateType(), True),
    StructField("policy_start_date", DateType(), True),
    StructField("policy_end_date", DateType(), True),
    StructField("renewal_date", DateType(), True),
    StructField("policy_status", StringType(), True),
    StructField("underwriting_status", StringType(), True),
    StructField("payment_mode_preference", StringType(), True),
    StructField("auto_renew_flag", BooleanType(), True),
    StructField("renewal_count", IntegerType(), True),
    StructField("created_at", TimestampType(), True)
])

In [0]:
enrollment_df = spark.read \
.option("header", "true") \
.schema(enrollment_schema) \
.csv(bronze_adls_path)

In [0]:
from pyspark.sql.functions import *

enrollment_df_silver = (
    enrollment_df

    # clean text
    .withColumn("policy_number", trim(col("policy_number")))
    .withColumn("policy_status", trim(col("policy_status")))
    .withColumn("underwriting_status", trim(col("underwriting_status")))
    .withColumn("payment_mode_preference", trim(col("payment_mode_preference")))

    # policy duration
    .withColumn(
        "policy_term_years",
        floor(datediff(col("policy_end_date"), col("policy_start_date")) / 365)
    )

    # policy age
    .withColumn(
        "policy_age_years",
        floor(datediff(current_date(), col("policy_start_date")) / 365)
    )

    # renewal expected flag
    .withColumn(
        "renewal_expected_flag",
        when(
            (col("auto_renew_flag") == True) &
            (col("policy_status") == "Active"),
            1
        ).otherwise(0)
    )

    # cancellation risk indicator
    .withColumn(
        "cancellation_risk_flag",
        when(
            (col("policy_status") == "Cancelled") |
            (col("underwriting_status") == "Manual Review"),
            1
        ).otherwise(0)
    )
)

In [0]:
from pyspark.sql.window import Window

window = Window.partitionBy(col("customer_id")).orderBy(col("created_at"))

enrollment_df_silver = (
    enrollment_df_silver
    .withColumn("rnk", rank().over(window))
    .filter(col("rnk")==1)
    .drop(col("rnk"))
)

In [0]:
enrollment_df_silver_final = enrollment_df_silver.select(

    # identifiers
    "customer_policy_id",
    "policy_number",
    "customer_id",
    "policy_product_id",

    # dates
    "enrollment_date",
    "policy_start_date",
    "policy_end_date",
    "renewal_date",

    # derived metrics
    "policy_term_years",
    "policy_age_years",

    # status
    "policy_status",
    "underwriting_status",

    # renewal info
    "auto_renew_flag",
    "renewal_expected_flag",
    "renewal_count",

    # payment
    "payment_mode_preference",

    # risk indicator
    "cancellation_risk_flag",

    # metadata
    "created_at"
)

enrollment_df_silver_final.write \
.format("delta") \
.mode("overwrite") \
.partitionBy("policy_start_date") \
.save(silver_adls_path)